Importing the libaries


In [ ]:
# Importing the necessary libraries

# Enable automatic reloading of modules when they are updated
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import os
import textwrap

PROJECT_ROOT = Path.cwd().resolve().parent
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from IPython.display import Image, display
from torchmetrics.text.bleu import BLEUScore

from src.train import (
    build_sequence_dataloaders,
    build_tokenizer,
    load_storyreasoning,
    train_experiment1,

   
)
from src.utils import (
    ensure_dirs,
    generate,
    load_config,
    set_seed,
    validation,
)


CONFIG_PATH = PROJECT_ROOT / "config.yaml"
config = load_config(str(CONFIG_PATH))

CONFIG_PATH = "config.yaml"
config = load_config(CONFIG_PATH)
set_seed(config.get("seed", 42))
ensure_dirs(config["paths"]["checkpoint_dir"], config["paths"]["results_dir"])
output_dir = Path(config["paths"]["results_dir"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")


Loading and Saving Data

In [ ]:
# Loading the dataset
tokenizer = build_tokenizer()
train_dataset, test_dataset = load_storyreasoning(config)
train_dataloader, val_dataloader, test_dataloader = build_sequence_dataloaders(
    config,
    tokenizer,
    train_dataset,
    test_dataset,
)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Train batches: {len(train_dataloader)}")
print(f"Validation batches: {len(val_dataloader)}")
print(f"Test batches: {len(test_dataloader)}")


In [ ]:
"""
A sanity check cell to verify the data pipeline.
It grabs a single batch from the training dataset and prints the shapes of the returned tensors (images, descriptions, etc.) to ensure everything is loaded correctly.
"""

frames, descriptions, image_target, text_target, roi1, roi2, roi_valid, roi_frame, ent_id = next(iter(train_dataloader))

print("frames:", frames.shape)
print("descriptions:", descriptions.shape)
print("image_target:", image_target.shape)
print("text_target:", text_target.shape)
print("roi_valid:", roi_valid.shape)


Experiment 1 - Grounding Module: Cross-Modal Attention


In [ ]:
experiment_name = "Experiment_1"
experiment1_dir = Path("results") / experiment_name
output_dir = experiment1_dir
ensure_dirs(experiment1_dir)
print(f"Experiment 1 outputs: {experiment1_dir}")

Training


In [ ]:
sequence_predictor, tokenizer, val_dataloader, losses, training_log = train_experiment1(
    CONFIG_PATH,
    show_validation=True,
) 

Saving the Training Logs

In [ ]:
experiment1_dir = Path("results") / "Experiment_1"
output_dir = experiment1_dir
ensure_dirs(experiment1_dir)
log_path = experiment1_dir / "training_log.txt"

with open(log_path, "w", encoding="utf-8") as f:
    for line in training_log:
        print(line)
        f.write(line + "\n")

print(f"Training log saved: {log_path}")

Validation Run

In [ ]:
validation(
    sequence_predictor,
    val_dataloader,
    tokenizer,
    device,
    show=True,
)
sequence_predictor.eval()

Image Output Visualization

In [ ]:
pred_path = experiment1_dir / "predictionexample.png"
gt_path = experiment1_dir / "groundtruth.png"
comparison_path = experiment1_dir / "visualcomparison.png"

sequence_predictor.eval()
frames, descriptions, image_target, text_target, *_ = next(iter(val_dataloader))

# moved data to device (GPU/CPu)
frames = frames.to(device)
descriptions = descriptions.to(device)
image_target = image_target.to(device)
text_target = text_target.to(device)

with torch.no_grad():
    # forward pass , generates predictied image and hidden states
    pred_img, _, _, h0, c0, _, _ = sequence_predictor(frames, descriptions, text_target)
    
    # generate predicted text sequence using decoder
    generated_tokens = generate(
        sequence_predictor.text_decoder,
        h0[:, 0, :].unsqueeze(1),
        c0[:, 0, :].unsqueeze(1),
        max_len=150,
        sos_token_id=tokenizer.cls_token_id,
        eos_token_id=tokenizer.sep_token_id,
        device=device,
    )

# handling dimension mismatch
if text_target.dim() == 3:
    text_target_decode = text_target.squeeze(1)
else:
    text_target_decode = text_target

true_sentence = tokenizer.decode(text_target_decode[0].cpu(), skip_special_tokens=True)
pred_sentence = tokenizer.decode(generated_tokens, skip_special_tokens=True)

plt.imsave(pred_path, pred_img[0].detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy())
plt.imsave(gt_path, image_target[0].detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy())

# Side by Side comparision figure
plt.figure(figsize=(10, 5))

# Ground truth 
plt.subplot(1, 2, 1)
plt.imshow(image_target[0].detach().cpu().clamp(0, 1).permute(1, 2, 0))
plt.title("Ground Truth (Target)")
plt.axis("off")

# Predicted Image
plt.subplot(1, 2, 2)
plt.imshow(pred_img[0].detach().cpu().clamp(0, 1).permute(1, 2, 0))
plt.title("Experiment 1 Prediction")
plt.axis("off")

plt.tight_layout()
plt.savefig(comparison_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()

print("Target Text:", true_sentence)
print("Predicted Text:", pred_sentence)
print(f"Saved: {pred_path}, {gt_path} and {comparison_path}") 

Generated Text Example

In [ ]:
sequence_predictor.eval()
frames, descriptions, image_target, text_target, *_ = next(iter(val_dataloader))
frames = frames.to(device)
descriptions = descriptions.to(device)
text_target = text_target.to(device)

with torch.no_grad():
    _, _, _, h0, c0, _, _ = sequence_predictor(frames, descriptions, text_target)
    generated_tokens = generate(
        sequence_predictor.text_decoder,
        h0[:, 0, :].unsqueeze(1),
        c0[:, 0, :].unsqueeze(1),
        max_len=150,
        sos_token_id=tokenizer.cls_token_id,
        eos_token_id=tokenizer.sep_token_id,
        device=device,
    )

if text_target.dim() == 3:
    text_target_decode = text_target.squeeze(1)
else:
    text_target_decode = text_target

true_sentence = tokenizer.decode(text_target_decode[0].cpu(), skip_special_tokens=True)
pred_sentence = tokenizer.decode(generated_tokens, skip_special_tokens=True)

prediction_text_path = experiment1_dir / "prediction_text_example.txt"
with open(prediction_text_path, "w", encoding="utf-8") as f:
    f.write("Experiment 1 Prediction Example\n")
    f.write("=" * 40 + "\n")
    f.write(f"Target Text: {true_sentence}\n")
    f.write(f"Predicted Text: {pred_sentence}\n")

print("Target Text:", true_sentence)
print("Predicted Text:", pred_sentence)
print(f"Prediction text example saved: {prediction_text_path}")


Attention HeatMap for Experiment

Attention Data

In [ ]:
attention_matrix_path = experiment1_dir / "attentionheatmap.png"
attention_overlay_path = experiment1_dir / "attention_overlay.png"

sequence_predictor.eval()
with torch.no_grad():
    frames, descriptions, image_target, text_target, *_ = next(iter(val_dataloader))
    frames = frames.to(device)
    descriptions = descriptions.to(device)
    text_target = text_target.to(device)

    sequence_predictor(frames, descriptions, text_target)

cross_attention = sequence_predictor.last_cross_attention
attention_hw = sequence_predictor.last_cross_attention_hw

if cross_attention is None or attention_hw is None:
    raise RuntimeError("Cross-Modal Attention weights were not found. Run Experiment 1 before this cell.")

sample_index = 0
frame_count = min(4, frames.size(1))
attn_h, attn_w = attention_hw

sample_attention = cross_attention[sample_index, :frame_count].detach().cpu()
sample_token_ids = descriptions[sample_index, :frame_count].detach().cpu()

ignore_ids = {tokenizer.pad_token_id, tokenizer.cls_token_id, tokenizer.sep_token_id}
token_mask = torch.ones_like(sample_token_ids, dtype=torch.bool)
for token_id in ignore_ids:
    if token_id is not None:
        token_mask &= sample_token_ids != token_id

print("Experiment 1 attention data prepared.")
print("Attention feature-map size:", attention_hw)


Attention Matrix Heatmap

In [ ]:
matrix_rows = []
for frame_index in range(frame_count):
    valid_attention = sample_attention[frame_index][token_mask[frame_index]]
    if valid_attention.numel() == 0:
        valid_attention = sample_attention[frame_index]

    frame_region_attention = valid_attention.mean(dim=0).view(attn_h, attn_w)
    pooled_regions = F.adaptive_avg_pool2d(
        frame_region_attention.unsqueeze(0).unsqueeze(0),
        output_size=(4, 4),
    ).squeeze()
    matrix_rows.append(pooled_regions.flatten())

attention_grid = torch.stack(matrix_rows)
attention_grid = (attention_grid - attention_grid.min(dim=1, keepdim=True).values) / (
    attention_grid.max(dim=1, keepdim=True).values
    - attention_grid.min(dim=1, keepdim=True).values
    + 1e-8
)

plt.figure(figsize=(12, 4.8))
heatmap = plt.imshow(attention_grid, cmap="YlOrRd", aspect="auto", vmin=0.0, vmax=1.0)
plt.title("Experiment 1: Cross-Modal Attention Heatmap")
plt.xlabel("Visual Region")
plt.ylabel("Story Frame")
plt.xticks(
    range(attention_grid.size(1)),
    [f"R{i + 1}" for i in range(attention_grid.size(1))],
    rotation=45,
    ha="right",
)
plt.yticks(range(frame_count), [f"Frame {i + 1}" for i in range(frame_count)])

for row in range(attention_grid.size(0)):
    for col in range(attention_grid.size(1)):
        value = attention_grid[row, col].item()
        text_color = "white" if value > 0.55 else "black"
        plt.text(col, row, f"{value:.2f}", ha="center", va="center", fontsize=8, color=text_color)

plt.colorbar(heatmap, fraction=0.035, pad=0.02, label="Attention Strength")
plt.tight_layout()
plt.savefig(attention_matrix_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print(f"Attention matrix saved: {attention_matrix_path}")


Attention Overlay Map

In [ ]:
frame_focus_scores = []
for frame_index in range(frame_count):
    valid_attention = sample_attention[frame_index][token_mask[frame_index]]
    if valid_attention.numel() == 0:
        valid_attention = sample_attention[frame_index]
    frame_focus_scores.append(valid_attention.max(dim=1).values.mean())

selected_frame_index = int(torch.stack(frame_focus_scores).argmax().item())
selected_token_ids = sample_token_ids[selected_frame_index]
selected_attention = sample_attention[selected_frame_index]
selected_valid_positions = torch.where(token_mask[selected_frame_index])[0]

if len(selected_valid_positions) > 0:
    candidate_attention = selected_attention[selected_valid_positions]
    token_focus_scores = candidate_attention.max(dim=1).values
    selected_token_position = selected_valid_positions[int(token_focus_scores.argmax().item())]
    selected_token = tokenizer.decode([int(selected_token_ids[selected_token_position])], skip_special_tokens=True).strip()
    spatial_attention = selected_attention[selected_token_position].view(attn_h, attn_w)
else:
    selected_token = "all tokens"
    spatial_attention = selected_attention.mean(dim=0).view(attn_h, attn_w)

spatial_attention = F.interpolate(
    spatial_attention.unsqueeze(0).unsqueeze(0),
    size=frames.shape[-2:],
    mode="bilinear",
    align_corners=False,
).squeeze()
spatial_attention = (spatial_attention - spatial_attention.min()) / (
    spatial_attention.max() - spatial_attention.min() + 1e-8
)

original_image = frames[sample_index, selected_frame_index].detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy()
attention_map = spatial_attention.detach().cpu().numpy()
decoded_text = tokenizer.decode(selected_token_ids.tolist(), skip_special_tokens=True)

figure, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(original_image)
axes[0].set_title(f"Original frame {selected_frame_index + 1}")
axes[0].axis("off")

heat = axes[1].imshow(attention_map, cmap="jet", vmin=0.0, vmax=1.0)
axes[1].set_title(f"Attention map: {selected_token or 'token'}")
axes[1].axis("off")
figure.colorbar(heat, ax=axes[1], fraction=0.046, pad=0.04)

axes[2].imshow(original_image)
axes[2].imshow(attention_map, cmap="jet", alpha=0.45, vmin=0.0, vmax=1.0)
axes[2].set_title("Overlay explanation map")
axes[2].axis("off")

figure.suptitle(textwrap.fill(decoded_text, width=95), fontsize=9)
figure.tight_layout()
figure.savefig(attention_overlay_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print(f"Selected frame: {selected_frame_index + 1}")
print(f"Selected token: {selected_token or 'token'}")
print(f"Attention overlay saved: {attention_overlay_path}")


Loss Curve

In [ ]:
plot_path = experiment1_dir / "losscurve.png"

plt.figure(figsize=(8, 5))
plt.plot(losses, label="Experiment 1 Training Loss", color="blue", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Experiment 1: Loss Curve")
plt.legend()
plt.grid(True)

plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()

print(f"Loss Curve Saved: {plot_path}")

Generate Predictions

In [ ]:
experiment1_dir = Path("results") / "Experiment_1"
output_dir = experiment1_dir
ensure_dirs(experiment1_dir)

sequence_predictor.eval()
pred_sentences = []
true_sentences = []
max_batches = config.get("evaluation", {}).get("bleu_max_batches", 5)

with torch.no_grad():
    for batch_index, (frames, descriptions, image_target, text_target, *_ ) in enumerate(val_dataloader):
        if max_batches is not None and batch_index >= max_batches:
            break

        frames = frames.to(device)
        descriptions = descriptions.to(device)
        text_target = text_target.to(device)

        _, _, _, h0, c0, _, _ = sequence_predictor(frames, descriptions, text_target)

        for i in range(frames.size(0)):
            generated_tokens = generate(
                sequence_predictor.text_decoder,
                h0[:, i, :].unsqueeze(1),
                c0[:, i, :].unsqueeze(1),
                max_len=150,
                sos_token_id=tokenizer.cls_token_id,
                eos_token_id=tokenizer.sep_token_id,
                device=device,
            )
            pred_sentences.append(tokenizer.decode(generated_tokens, skip_special_tokens=True))

        if text_target.dim() == 3:
            text_target_decode = text_target.squeeze(1)
        else:
            text_target_decode = text_target

        for seq in text_target_decode:
            true_sentences.append(tokenizer.decode(seq.cpu().numpy(), skip_special_tokens=True))

print("Number of predictions:", len(pred_sentences))
print("Number of true sentences:", len(true_sentences))


Calculate and Save Metrics

In [ ]:
reference = [[s] for s in true_sentences]
bleu1_metric = BLEUScore(n_gram=1)
bleu4_metric = BLEUScore(n_gram=4)
experiment1_bleu_val = bleu1_metric(pred_sentences, reference).item()
experiment1_bleu4_val = bleu4_metric(pred_sentences, reference).item()


def simple_meteor_score(prediction, reference_sentence):
    pred_tokens = prediction.lower().split()
    ref_tokens = reference_sentence.lower().split()
    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    pred_counts = {}
    ref_counts = {}
    for token in pred_tokens:
        pred_counts[token] = pred_counts.get(token, 0) + 1
    for token in ref_tokens:
        ref_counts[token] = ref_counts.get(token, 0) + 1

    matches = sum(min(pred_counts.get(token, 0), ref_counts.get(token, 0)) for token in pred_counts)
    if matches == 0:
        return 0.0

    precision = matches / len(pred_tokens)
    recall = matches / len(ref_tokens)
    return (10 * precision * recall) / (recall + 9 * precision + 1e-8)


try:
    from nltk.translate.meteor_score import meteor_score

    meteor_values = [
        meteor_score([true.lower().split()], pred.lower().split())
        for pred, true in zip(pred_sentences, true_sentences)
    ]
except Exception:
    meteor_values = [
        simple_meteor_score(pred, true)
        for pred, true in zip(pred_sentences, true_sentences)
    ]

experiment1_meteor_val = sum(meteor_values) / len(meteor_values) if meteor_values else 0.0

print("\n" + "=" * 50)
print("Experiment 1 Results")
print("=" * 50)
print(f"Final Training Loss: {losses[-1]:.4f}")
print(f"BLEU Score: {experiment1_bleu_val:.4f}")
print(f"BLEU-4 Score: {experiment1_bleu4_val:.4f}")
print(f"METEOR Score: {experiment1_meteor_val:.4f}")
print("=" * 50 + "\n")

metrics_path = experiment1_dir / "metrics.txt"
with open(metrics_path, "w", encoding="utf-8") as f:
    f.write("Experiment 1\n")
    f.write("=" * 40 + "\n")
    f.write(f"{'Metric':<25} | {'Value':<10}\n")
    f.write("-" * 40 + "\n")
    f.write(f"{'Final Training Loss':<25} | {losses[-1]:.4f}\n")
    f.write(f"{'BLEU Score':<25} | {experiment1_bleu_val:.4f}\n")
    f.write(f"{'BLEU-4 Score':<25} | {experiment1_bleu4_val:.4f}\n")
    f.write(f"{'METEOR Score':<25} | {experiment1_meteor_val:.4f}\n")
    f.write(f"{'Epochs Completed':<25} | {len(losses)}\n")
    f.write(f"{'Predictions Evaluated':<25} | {len(pred_sentences)}\n")

print(f"Experiment 1 metrics table saved: {metrics_path}")


Comparison with Baseline

In [ ]:
baseline_dir = Path("results") / "baseline"
experiment1_dir = Path("results") / "Experiment_1"
comparison_path = experiment1_dir / "comparison_table.txt"

def load_metrics_table(metrics_path):
    metrics = {}
    with open(metrics_path, "r", encoding="utf-8") as f:
        for line in f:
            if "|" in line and "Metric" not in line:
                name, value = line.split("|", 1)
                name = name.strip()
                value = value.strip()
                try:
                    metrics[name] = float(value)
                except ValueError:
                    metrics[name] = value
    return metrics

baseline_metrics = load_metrics_table(baseline_dir / "metrics.txt")
experiment1_metrics = load_metrics_table(experiment1_dir / "metrics.txt")

comparison_rows = [
    "Experiment 1 vs Baseline",
    "=" * 68,
    f"{'Metric':<25} | {'Baseline':<12} | {'Experiment 1':<12} | {'Change':<10}",
    "-" * 68,
]

for metric in [
    "Final Training Loss",
    "BLEU Score",
    "BLEU-4 Score",
    "METEOR Score",
    "Epochs Completed",
    "Predictions Evaluated",
]:
    baseline_value = baseline_metrics.get(metric, 0.0)
    experiment1_value = experiment1_metrics.get(metric, 0.0)
    if isinstance(baseline_value, float) and isinstance(experiment1_value, float):
        change = experiment1_value - baseline_value
        comparison_rows.append(
            f"{metric:<25} | {baseline_value:<12.4f} | {experiment1_value:<12.4f} | {change:<10.4f}"
        )
    else:
        comparison_rows.append(
            f"{metric:<25} | {baseline_value!s:<12} | {experiment1_value!s:<12} | {'-':<10}"
        )

comparison_text = "\n".join(comparison_rows)
print(comparison_text)

with open(comparison_path, "w", encoding="utf-8") as f:
    f.write(comparison_text + "\n")

print(f"Experiment 1 comparison table saved: {comparison_path}")
